In [ ]:

%pip install -q -U crewai crewai-tools ddgs


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 238.9/238.9 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 71.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.5

In [ ]:

import os
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError(
        "GEMINI_API_KEY not found. Add it in Colab Secrets."
    )

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

print("Gemini API key loaded successfully.")


Gemini API key loaded successfully.


In [ ]:
import os
import csv
import json
from pathlib import Path
from datetime import datetime, timezone

from pydantic import BaseModel, Field
from crewai import Agent, Task, Crew, Process, LLM
from crewai.flow.flow import Flow, start, listen, router, or_
from crewai_tools import ScrapeWebsiteTool

llm = LLM(
    model="gemini/gemini-3.1-flash-lite-preview",
    api_key=GEMINI_API_KEY,
    temperature=0.4
)

OUTPUT_DIR = Path("/content/lead_qualification_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Imports and configuration completed.")
print("Output directory:", OUTPUT_DIR)

Imports and configuration completed.
Output directory: /content/lead_qualification_output


In [ ]:
class LeadQualificationState(BaseModel):
    lead_name: str = ""
    website: str = ""
    website_content: str = ""
    enrichment: str = ""
    category: str = ""
    reasoning: str = ""
    action: str = ""
    outreach_email: str = ""
    nurture_tips: list[str] = Field(default_factory=list)
    error: str = ""


def save_json(filename, data):
    path = OUTPUT_DIR / filename
    with open(path, "w", encoding="utf-8") as file:
        json.dump(data, file, indent=2, ensure_ascii=False)
    return str(path)


def append_csv(filename, row, fieldnames):
    path = OUTPUT_DIR / filename
    file_exists = path.exists() and path.stat().st_size > 0

    with open(path, "a", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)

    return str(path)


def run_crew(agent_role, goal, backstory, task_description, expected_output):
    agent = Agent(
        role=agent_role,
        goal=goal,
        backstory=backstory,
        llm=llm,
        verbose=False
    )

    task = Task(
        description=task_description,
        expected_output=expected_output,
        agent=agent
    )

    crew = Crew(
        agents=[agent],
        tasks=[task],
        process=Process.sequential,
        verbose=False
    )

    return str(crew.kickoff())

In [ ]:
class LeadQualificationFlow(Flow[LeadQualificationState]):

    @start()
    def capture_lead(self):
        print("\n=== LEAD CAPTURE ===")

        self.state.lead_name = input("Enter lead/company name: ").strip()
        self.state.website = input(
            "Enter website URL (include https://): "
        ).strip()

        if not self.state.lead_name:
            raise ValueError("Lead name cannot be empty.")

        if not self.state.website.startswith(("https://", "http://")):
            raise ValueError(
                "Enter a complete website URL beginning with https:// or http://"
            )

        return {
            "lead_name": self.state.lead_name,
            "website": self.state.website
        }

    @listen(capture_lead)
    def enrich_lead(self, lead_data):
        print("\n=== WEBSITE ENRICHMENT ===")

        try:
            scraper = ScrapeWebsiteTool()

            clean_url = self.state.website.strip()

            # Handle a URL accidentally entered as a Markdown link
            if clean_url.startswith("[") and "](" in clean_url:
             clean_url = clean_url.split("](", 1)[1].rstrip(")")

            self.state.website = clean_url

            scraped_content = scraper.run(website_url=clean_url)

            self.state.website_content = str(scraped_content)[:10000]


            self.state.enrichment = run_crew(
                agent_role="Lead Research Analyst",
                goal="Summarize the lead using evidence from its website.",
                backstory=(
                    "You are a careful B2B research analyst. "
                    "You distinguish facts from assumptions."
                ),
                task_description=f"""
                Analyze this prospective lead.

                Company/lead name: {self.state.lead_name}
                Website: {self.state.website}

                Website content:
                {self.state.website_content}

                Extract:
                1. What the organization does.
                2. Its products or services.
                3. Its likely target customers.
                4. Any evidence relevant to B2B qualification.
                5. Information that is missing or uncertain.

                Use only the supplied website content.
                Do not invent revenue, budget, purchasing intent, or contacts.
                """,
                expected_output=(
                    "A concise, evidence-based company profile with "
                    "clear notes about missing information."
                )
            )

        except Exception as e:
            self.state.website_content = ""
            self.state.enrichment = (
                "Website enrichment failed. Manual verification is required. "
                f"Error: {str(e)[:500]}"
            )
            self.state.error = str(e)[:500]

        save_json("enriched_lead.json", {
            "lead_name": self.state.lead_name,
            "website": self.state.website,
            "website_content": self.state.website_content,
            "enrichment": self.state.enrichment,
            "error": self.state.error
        })

        print("Enrichment stage completed.")
        return self.state.enrichment

    @router(enrich_lead)
    def score_lead(self):
        print("\n=== LEAD SCORING ===")

        result = run_crew(
            agent_role="Lead Qualification Specialist",
            goal="Classify leads fairly using available evidence.",
            backstory=(
                "You qualify leads for a sales team. "
                "You never treat missing information as proof of low interest."
            ),
            task_description=f"""
            Classify the following lead as exactly HOT, WARM, or COLD.

            Lead: {self.state.lead_name}
            Website: {self.state.website}
            Website analysis: {self.state.enrichment}

            Rules:
            - HOT: Strong, specific evidence of relevant fit and a credible
              near-term buying signal.
            - WARM: Potential fit, but intent or key qualification details
              are missing or need follow-up.
            - COLD: Clear evidence of poor fit against the configured
              business context, not merely missing website information.

            No target customer profile or product offering has been provided.
            Therefore, do not invent one. If fit or buying intent cannot be
            established from the available evidence, classify as WARM.

            Return exactly this structure:
            CATEGORY: HOT, WARM, or COLD
            REASON: concise evidence-based explanation
            """,
            expected_output=(
                "A CATEGORY line containing HOT, WARM, or COLD and a REASON line."
            )
        )

        upper_result = result.upper()

        if "CATEGORY: HOT" in upper_result:
            self.state.category = "hot"
        elif "CATEGORY: COLD" in upper_result:
            self.state.category = "cold"
        else:
            self.state.category = "warm"

        self.state.reasoning = result

        print("Classification:", self.state.category.upper())
        print("Reasoning:", result)

        return self.state.category

    @listen("hot")
    def hot_outreach(self):
        print("\n=== HOT LEAD: OUTREACH CREW ===")

        self.state.outreach_email = run_crew(
            agent_role="Personalized Sales Outreach Writer",
            goal="Draft a relevant and respectful first-contact email.",
            backstory=(
                "You write concise B2B emails grounded in verified company "
                "information and avoid fabricated personalization."
            ),
            task_description=f"""
            Draft an email for this lead.

            Lead: {self.state.lead_name}
            Website: {self.state.website}
            Research: {self.state.enrichment}
            Qualification result: {self.state.reasoning}

            Requirements:
            - Include a subject line, greeting, short body, and sign-off.
            - Personalize only with supported facts.
            - Do not claim that the lead requested contact.
            - Do not invent the sender's name, product, results, or pricing.
            - Use placeholders where sender/product details are unknown.
            - Make the email professional and low-pressure.
            """,
            expected_output="A complete email draft with a subject line."
        )

        self.state.action = "Drafted personalized outreach email"

        save_json("hot_lead_outreach.json", {
            "lead_name": self.state.lead_name,
            "website": self.state.website,
            "category": self.state.category,
            "reasoning": self.state.reasoning,
            "email_draft": self.state.outreach_email
        })

        print(self.state.outreach_email)
        return self.state.action

    @listen("warm")
    def add_to_nurture(self):
        print("\n=== WARM LEAD: NURTURE CREW ===")

        result = run_crew(
            agent_role="Lead Nurture Content Planner",
            goal="Suggest useful educational content for a potential lead.",
            backstory=(
                "You create helpful, non-pushy B2B educational content."
            ),
            task_description=f"""
            Create three weekly educational tips or content ideas for this lead.

            Lead: {self.state.lead_name}
            Website: {self.state.website}
            Research: {self.state.enrichment}

            Return exactly three numbered tips.
            Keep them relevant to verified information where possible.
            If the lead's needs are unclear, make the uncertainty explicit.
            Do not claim that the lead consented to receive marketing emails.
            Use only information supported by the supplied research.
            If website enrichment failed, do not invent company-specific needs,
            current initiatives, technology choices, or business priorities.
            Generate general educational ideas and explicitly state that
            the company's specific needs remain unverified.
            """,
            expected_output="Three numbered educational tips."
        )

        self.state.nurture_tips = [
            line.strip()
            for line in result.splitlines()
            if line.strip()
        ][:3]

        self.state.action = "Added lead to nurture list"

        append_csv(
            "nurture_list.csv",
            {
                "lead_name": self.state.lead_name,
                "website": self.state.website,
                "reasoning": self.state.reasoning,
                "weekly_tips": " | ".join(self.state.nurture_tips)
            },
            ["lead_name", "website", "reasoning", "weekly_tips"]
        )

        print(result)
        return self.state.action

    @listen("cold")
    def archive_lead(self):
        print("\n=== COLD LEAD: ARCHIVE ===")

        self.state.action = "Archived for review; no outreach drafted"

        append_csv(
            "archived_leads.csv",
            {
                "lead_name": self.state.lead_name,
                "website": self.state.website,
                "reasoning": self.state.reasoning,
                "archived_at": datetime.now(timezone.utc).isoformat()
            },
            ["lead_name", "website", "reasoning", "archived_at"]
        )

        print("Lead archived. No outreach was generated.")
        return self.state.action

    @listen(or_(hot_outreach, add_to_nurture, archive_lead))
    def log_lead(self):
        print("\n=== LOGGING LEAD ===")

        append_csv(
            "leads.csv",
            {
                "lead_name": self.state.lead_name,
                "website": self.state.website,
                "category": self.state.category,
                "reasoning": self.state.reasoning,
                "action": self.state.action,
                "logged_at": datetime.now(timezone.utc).isoformat()
            },
            [
                "lead_name",
                "website",
                "category",
                "reasoning",
                "action",
                "logged_at"
            ]
        )

        print("Lead saved to:", OUTPUT_DIR / "leads.csv")

        return {
            "lead_name": self.state.lead_name,
            "website": self.state.website,
            "category": self.state.category,
            "action": self.state.action
        }

In [ ]:
flow = LeadQualificationFlow()
result = flow.kickoff()

print("\n=== FINAL RESULT ===")
print(result)

╭─────────────────────────────────────────────── 🌊 Flow Execution ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name: LeadQualificationFlow                                                                                    │
│  ID: 4f9769fe-ec3c-41c9-8cb5-818f41b15d22                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🌊 Flow Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Started                                                                                                   │
│  Name: LeadQualificationFlow                                                                                    │
│  ID: 4f9769fe-ec3c-41c9-8cb5-818f41b15d22                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


=== LEAD CAPTURE ===


╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: capture_lead                                                                                           │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Enter lead/company name: stripe
Enter website URL (include https://): https://stripe.com

=== WEBSITE ENRICHMENT ===


╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: capture_lead                                                                                           │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: enrich_lead                                                                                            │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Enrichment stage completed.

=== LEAD SCORING ===


╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: score_lead                                                                                             │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: enrich_lead                                                                                            │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [ ]:
from pathlib import Path
import pandas as pd

OUTPUT_DIR = Path("/content/lead_qualification_output")

print("CSV files generated:\n")

for path in sorted(OUTPUT_DIR.glob("*.csv")):
    print(f"\n--- {path.name} ---")
    try:
        df = pd.read_csv(path)
        display(df)
    except Exception as e:
        print(f"Could not read {path.name}: {e}")

Files generated:


--- enriched_lead.json ---
{
  "lead_name": "stripe",
  "website": "https://stripe.com",
  "website_content": "",
  "enrichment": "Website enrichment failed. Manual verification is required. Error: Tool 'Read website content' arguments validation failed: 1 validation error for ScrapeWebsiteToolSchema\nwebsite_url\n  Field required [type=missing, input_value={'url': 'https://stripe.com'}, input_type=dict]\n    For further information visit https://errors.pydantic.dev/2.12/v/missing\nExpected arguments: {\"website_url\": {\"description\": \"Mandatory website url to read the file\", \"title\": \"Website Url\", \"type\": \"string\"}}\nRequired: [\"website_url\"]",
  "error": "Tool 'Read website content' arguments validation failed: 1 validation error for ScrapeWebsiteToolSchema\nwebsite_url\n  Field required [type=missing, input_value={'url': 'https://stripe.com'}, input_type=dict]\n    For further information visit https://errors.pydantic.dev/2.12/v/missing\nExpected ar

In [ ]:
from pathlib import Path

OUTPUT_DIR = Path("/content/lead_qualification_output")

for filename in [
    "leads.csv",
    "nurture_list.csv",
    "archived_leads.csv",
    "enriched_lead.json",
    "hot_lead_outreach.json",
    "enriched_lead.csv",
    "hot_lead_outreach.csv"
]:
    path = OUTPUT_DIR / filename
    if path.exists():
        path.unlink()

print("Previous demo outputs cleared.")

Previous demo outputs cleared.
